## Model

In [ ]:
import torch, gc

torch.cuda.set_device(7)
torch.cuda.empty_cache()
gc.collect()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_CLASSES = 20
IMG_SIZE = (224, 224)


In [ ]:
muse_dir = "/home/chenyinjia/data/dataset/muses"
ade20k_dir = "/home/chenyinjia/data/dataset/ADE20ID"
cs_dir = "/home/chenyinjia/data/dataset/cityscapes"
acdc_dir = "/home/chenyinjia/data/dataset/acdc"

clip32_dir = "/home/chenyinjia/clip-vit-base-patch32"
MODEL_PATH = "/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/checkpoints2/base_alpha.pth"  # checkpoint ที่เซฟไว้

clip32_dir = "/home/chenyinjia/clip-vit-large-patch14"
MODEL_PATH = "/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/checkpoints3/L14m6.pth"

In [ ]:
# ============================================================
# 🎯 CLIPSeg Inference with Overlay + Text Labels (no box)
# ============================================================

import torch
import torch.nn.functional as F
from transformers import CLIPModel
import clip
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch.nn as nn
import matplotlib.cm as cm

from blora import BlockLoraConfig, get_blocklora_model, BlockLoraModel
from peft import LoraConfig, get_peft_model, PeftModel

criterion = nn.CrossEntropyLoss(ignore_index=255)
# ====================================================
# 🏗️ แก้ไข OpenVocabSegModel (ดึง patch features)
# ====================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CLIPSegBaseline(nn.Module):
    def __init__(self, clip_model, num_classes=NUM_CLASSES):
        super().__init__()
        self.clip = clip_model.vision_model  # HF: CLIPVisionModel
        self.num_classes = num_classes
        
        # Freeze CLIP encoder
        for param in self.clip.parameters():
            param.requires_grad = False
        
        self.feature_dim = 1024  # usually 768
        
        # Decoder head
        self.decoder = nn.Sequential(
            nn.Conv2d(self.feature_dim, 512, 3, padding=1),
            nn.ReLU(),
            nn.Dropout2d(p=0.1),
            nn.Conv2d(512, 256, 3, padding=1),
            nn.ReLU(),
            nn.Dropout2d(p=0.05),
            nn.Conv2d(256, num_classes, 1)
        )
    
    def forward(self, x):
        B, _, H, W = x.shape
        seq_len = (H // self.clip.config.patch_size) * (W // self.clip.config.patch_size)  # 14*14=196

        with torch.no_grad():
            # Patch embedding
            x_vit = self.clip.embeddings.patch_embedding(x)  # [B, 768, 14, 14]
            x_vit = x_vit.flatten(2).transpose(1, 2)        # [B, 196, 768]

            # CLS token
            if hasattr(self.clip.embeddings, "cls_token"):
                cls_token = self.clip.embeddings.cls_token.expand(B, -1, -1)
            else:
                cls_token = torch.zeros(B, 1, self.feature_dim, device=x.device)
            x_vit = torch.cat([cls_token, x_vit], dim=1)  # [B, 197, 768]

            # Positional embeddings: index
            pos_ids = torch.arange(x_vit.shape[1], device=x.device)
            pos_embeds = self.clip.embeddings.position_embedding(pos_ids)  # [197, 768]
            x_vit = x_vit + pos_embeds.unsqueeze(0)  # broadcast to [B, 197, 768]

            # LayerNorm
            x_vit = self.clip.pre_layrnorm(x_vit)    
            # x_vit = self.clip.embeddings.ln_pre(x_vit)

            # Transformer
            x_vit = self.clip.encoder(x_vit)[0]
            features = x_vit[:, 1:, :].float()  # skip CLS token

        
        # Reshape spatial map 14x14
        features = features.reshape(B, H // self.clip.config.patch_size, W // self.clip.config.patch_size, self.feature_dim)
        features = features.permute(0, 3, 1, 2)  # [B, 768, 14, 14]
        
        # Decode
        seg_logits = self.decoder(features)  # [B, num_classes, 14, 14]
        
        # Upsample
        seg_logits = F.interpolate(seg_logits, size=(H, W), 
                                   mode='bilinear', align_corners=False)
        return seg_logits


In [ ]:
BlockLoraConfig().Cheer

## Dataset Loder

In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset
from PIL import Image

# ============================================================
# 🔧 ADE20K official decoder
# ============================================================
def ade20k_rgb_to_class(mask_rgb: np.ndarray):
    """Decode ADE20K RGB mask to class IDs (official MIT method)."""
    mask_id = (
        mask_rgb[:, :, 0].astype(np.int32)
        + 256 * mask_rgb[:, :, 1].astype(np.int32)
        + 256 * 256 * mask_rgb[:, :, 2].astype(np.int32)
    )
    class_id = mask_id % 256
    return class_id.astype(np.int64)

# --- Cityscapes label set (index reference) ---
cityscapes_labels = [
    "road", "sidewalk", "building", "wall", "fence", "pole", "traffic light", "traffic sign",
    "vegetation", "terrain", "sky", "person", "rider", "car", "truck", "bus", 
    "train", "motorcycle", "bicycle", "unknown"
]

# --- ADE20K → Cityscapes Unified Mapping ---
ADE20K_TO_CITYSCAPES = {
    0: 255,   # background → unknown
    1: 255,    # wall
    2: 2,    # building
    3: 10,   # sky
    4: 255,    # floor → unknown
    5: 8,    # tree → vegetation
    6: 255,   # ceiling → unknown
    7: 0,    # road
    8: 255,   # bed
    9: 255,   # windowpane
    10: 8,   # grass → vegetation
    11: 255,  # cabinet → unknown
    12: 1,   # sidewalk
    13: 11,  # person
    14: 9,   # earth → terrain
    15: 255,   # door → unknown
    16: 255,  # table
    17: 255,   # mountain → terrain
    18: 8,   # plant → vegetation
    19: 255,  # curtain
    20: 255,  # chair
    21: 13,  # car
    22: 255,   # water → terrain
    23: 255,  # painting
    24: 255,  # sofa
    25: 255,  # shelf
    26: 2,   # house → building
    27: 255,   # sea → terrain
    28: 255,  # mirror
    29: 255,   # rug → terrain
    30: 9,   # field → terrain
    31: 255,  # armchair
    32: 255,  # seat
    33: 4,   # fence
    34: 255,  # desk
    35: 255,   # rock → terrain
    36: 255,  # wardrobe
    37: 255,  # lamp
    38: 255,  # bathtub
    39: 4,   # railing → fence
    40: 255,  # pillow
    41: 255,  # base
    42: 255,  # box
    43: 5,   # column → pole
    44: 7,   # signboard → traffic sign
    45: 255,  # chest of drawers
    46: 255,  # counter
    47: 9,   # sand → terrain
    48: 255,  # sink
    49: 2,   # skyscraper → building
    50: 255,  # fireplace
    51: 255,  # refrigerator
    52: 255,  # grandstand
    53: 0,   # path → road
    54: 255,  # stairs
    55: 0,   # runway → road
    56: 255,  # case
    57: 255,  # pool table
    58: 255,  # pillow
    59: 255,   # screen door → terrain
    60: 255, # stairway
    61: 255,   # river → terrain
    62: 9,   # bridge → building
    63: 255,  # bookcase 
    64: 255,  # blind
    65: 255,  # coffee table
    66: 255,  # toilet
    67: 8,    # flower → vegetation
    68: 255,  # book
    69: 9,    # hill → terrain
    70: 255,  # bench
    71: 255,  # countertop
    72: 255,  # stove
    73: 8,    # palm → vegetation
    74: 255,  # kitchen island
    75: 255,  # computer
    76: 255,  # swivel chair
    77: 255,  # boat → bicycle (closest moving object type)
    78: 255,  # bar
    79: 255,  # arcade machine
    80: 2,   # hovel → building
    81: 15,  # bus
    82: 255,  # towel
    83: 255,  # light
    84: 14,  # truck
    85: 2,   # tower → building
    86: 255,  # chandelier
    87: 5,   # awning → pole
    88: 5,   # streetlight → pole
    89: 255,  # booth
    90: 255,  # television receiver
    91: 255,  # airplane → motorcycle (generic vehicle)
    92: 0,   # dirt track → road
    93: 255,  # apparel
    94: 5,   # pole
    95: 9,   # land → terrain
    96: 4,   # bannister → fence
    97: 255,  # escalator
    98: 255,  # ottoman
    99: 255,  # bottle
    100: 255, # buffet
    101: 255, # poster
    102: 255, # stage
    103: 15, # van → bus
    104: 255, # ship → bicycle (vehicle)
    105: 255,  # fountain → terrain
    106: 255, # conveyer belt
    107: 255, # canopy
    108: 255, # washer
    109: 255, # plaything
    110: 255,  # swimming pool → terrain
    111: 255, # stool
    112: 255, # barrel
    113: 255, # basket
    114: 255,  # waterfall → terrain
    115: 255, # tent
    116: 255, # bag
    117: 17, # minibike → motorcycle
    118: 255, # cradle
    119: 255, # oven
    120: 255, # ball
    121: 255, # food
    122: 255, # step
    123: 255, # tank (ambiguous)
    124: 255, # trade name
    125: 255, # microwave
    126: 255, # pot
    127: 11, # animal → person (animate)
    128: 18, # bicycle
    129: 255,  # lake → terrain
    130: 255, # dishwasher
    131: 255, # screen
    132: 255, # blanket
    133: 255, # sculpture
    134: 255, # hood
    135: 5,  # sconce → pole
    136: 255, # vase
    137: 6,  # traffic light
    138: 255, # tray
    139: 255, # ashcan
    140: 255, # fan
    141: 9,  # pier → terrain
    142: 255, # crt screen
    143: 255, # plate
    144: 255, # monitor
    145: 255, # bulletin board
    146: 255, # shower
    147: 255, # radiator
    148: 255, # glass
    149: 255, # clock
    150: 255, # flag
}


def convert_ade20k_to_cityscapes(mask: np.ndarray):
    """Convert ADE20K mask IDs (0–149) → Unified Cityscapes space (0–19)."""
    out = np.zeros_like(mask, dtype=np.int64)
    for k, v in ADE20K_TO_CITYSCAPES.items():
        out[mask == k] = v
    # print("Unique ADE mask before map:", np.unique(mask))
    # print("Unique after map:", np.unique(out))

    return out

# ============================================================
# 🔹 ADE20K Dataset Loader
# ============================================================
class ADE20KDataset(Dataset):
    def __init__(self, root_dir, filter20 = False ,split="train"):
        self.samples = []
        split_dir = {"train": "training", "val": "validation", "test": "validation"}[split]

        self.img_root = os.path.join(root_dir, "images", split_dir)
        self.mask_root = os.path.join(root_dir, "annotations", split_dir)

        if not os.path.exists(self.img_root):
            raise FileNotFoundError(f"❌ Missing folder: {self.img_root}")

        img_files = sorted([f for f in os.listdir(self.img_root) if f.endswith(".jpg")])
        mask_files = sorted([f for f in os.listdir(self.mask_root) if f.endswith(".png")])

        for img_file in img_files:
            base = img_file.replace(".jpg", "")
            seg_file = base + ".png"
            img_path = os.path.join(self.img_root, img_file)
            seg_path = os.path.join(self.mask_root, seg_file)

            # ---  20 Class filtering ---
            if filter20:
                mask_img = Image.open(seg_path)
                mask_rgb = np.array(mask_img.convert("RGB"), dtype=np.uint8)
                mask = ade20k_rgb_to_class(mask_rgb)
                mask = convert_ade20k_to_cityscapes(mask)

                unknown_ratio = np.mean(mask == 255)
                if unknown_ratio > 0.5:
                    continue
                
            if os.path.exists(seg_path):
                self.samples.append((img_path, seg_path))

        print(f"Loaded {len(self.samples)} ADE20K {split} samples")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]
        image = Image.open(img_path).convert("RGB").resize(IMG_SIZE)

        # decode ADE20K mask
        mask_img = Image.open(mask_path)
        # if mask_img.mode == "I":
        #     mask = np.array(mask_img, dtype=np.int32) % 256
        # else:
        mask_rgb = np.array(mask_img.convert("RGB"), dtype=np.uint8)
        mask = ade20k_rgb_to_class(mask_rgb)

        mask = Image.fromarray(mask.astype(np.uint8)).resize(IMG_SIZE, resample=Image.NEAREST)
        mask = np.array(mask, dtype=np.uint8)
        mask = convert_ade20k_to_cityscapes(mask)

        image = np.array(image).transpose(2, 0, 1) / 255.0

        image = torch.tensor(image, dtype=torch.float32)
        mask = torch.tensor(mask, dtype=torch.long)
        return image, mask


# ============================================================
# 🔹 Unified MUSES Dataset Loader (All Weather + Day/Night)
# ============================================================
class MusesDataset(Dataset):
    def __init__(self, root_dir, weather="all",split="train"):
        self.root = root_dir
        self.split = split
        self.samples = []

        # sub-conditions ทั้งหมด เช่น clear/day, fog/night, rain/day ...
        all_weathers = ["clear", "fog", "rain", "snow"]
        if weather == "all":
            weathers = all_weathers
        elif isinstance(weather, (list, tuple)):
            weathers = [w for w in weather if w in all_weathers]
        elif weather in all_weathers:
            weathers = [weather]
        else:
            raise ValueError(f"Invalid weather '{weather}'. Choose from {all_weathers + ['all']}")
        # print(f"✅ Loading MUSES dataset with weathers: {weathers}")
        times_of_day = ["day", "night"]

        frame_root = os.path.join(root_dir, "frame_camera", split)
        mask_root = os.path.join(root_dir, "gt_semantic", split)

        for w in weathers:
            for tod in times_of_day:
                img_dir = os.path.join(frame_root, w, tod)
                mask_dir = os.path.join(mask_root, w, tod)

                if not (os.path.exists(img_dir) and os.path.exists(mask_dir)):
                    print(f"⚠️ Skip missing folder: {img_dir} or {mask_dir}")
                    continue

                img_files = sorted([f for f in os.listdir(img_dir) if f.endswith(".png")])
                mask_files = sorted([f for f in os.listdir(mask_dir) if f.endswith("_gt_labelTrainIds.png")])

                for img_file in img_files:
                    prefix = img_file.replace("_frame_camera.png", "")
                    match = [m for m in mask_files if prefix in m]
                    if match:
                        self.samples.append((os.path.join(img_dir, img_file),
                                             os.path.join(mask_dir, match[0])))

        print(f"Loaded {len(self.samples)} MUSES {split} samples with weathers: {weathers}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]
        image = Image.open(img_path).convert("RGB").resize(IMG_SIZE)
        mask = Image.open(mask_path).convert("L").resize(IMG_SIZE, resample=Image.NEAREST)

        image = np.array(image).transpose(2, 0, 1) / 255.0
        mask = np.array(mask, dtype=np.uint8)

        return torch.tensor(image, dtype=torch.float32), torch.tensor(mask, dtype=torch.long)



# ============================================================
# 🔹 Cityscapes Dataset Loader (Final Verified)
# ============================================================
class CityscapesDataset(Dataset):
    def __init__(self, root_dir, split="train"):
        self.split = split
        self.samples = []

        self.img_root = os.path.join(root_dir, "leftImg8bit", split)
        self.mask_root = os.path.join(root_dir, "gtFine", split)

        if not os.path.exists(self.img_root):
            raise FileNotFoundError(f"❌ Missing folder: {self.img_root}")

        for city in os.listdir(self.img_root):
            img_dir = os.path.join(self.img_root, city)
            mask_dir = os.path.join(self.mask_root, city)
            if not os.path.exists(mask_dir):
                continue

            for f in os.listdir(img_dir):
                if f.endswith("_leftImg8bit.png"):
                    base = f.replace("_leftImg8bit.png", "")
                    # ✅ fallback: ถ้าไม่มี labelTrainIds ให้ใช้ labelIds
                    mask_file = None
                    for suffix in ["_gtFine_labelTrainIds.png"] : # , "_gtFine_labelIds.png"]
                        candidate = os.path.join(mask_dir, base + suffix)
                        if os.path.exists(candidate):
                            mask_file = candidate
                            break
                    if mask_file:
                        self.samples.append((os.path.join(img_dir, f), mask_file))

        print(f"Loaded {len(self.samples)} Cityscapes {split} samples")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]
        image = Image.open(img_path).convert("RGB").resize(IMG_SIZE)
        mask = Image.open(mask_path).convert("L").resize(IMG_SIZE, 
                                                         resample=Image.NEAREST)
        image = np.array(image).transpose(2, 0, 1) / 255.0
        mask = np.array(mask, dtype=np.uint8)
        return torch.tensor(image, dtype=torch.float32), torch.tensor(mask, dtype=torch.long)


# ============================================================
# 🔹 ACDC Dataset Loader (Multiple Weathers)
# ============================================================
class ACDCDataset(Dataset):
    """
    ACDC dataset loader supporting multiple weathers (fog/night/rain/snow)
    Structure expected:
        dataset/acdc/{rgb,gt}/{weather}/{split}/{video}/
    """
    def __init__(self, root_dir, weather="all", split="train"):
        self.split = split
        self.samples = []

        # --- weather selection ---
        all_weathers = ["fog", "night", "rain", "snow"]
        if weather == "all":
            weathers = all_weathers
        elif isinstance(weather, (list, tuple)):
            weathers = [w for w in weather if w in all_weathers]
        elif weather in all_weathers:
            weathers = [weather]
        else:
            raise ValueError(f"Invalid weather '{weather}'. Choose from {all_weathers + ['all']}")

        # --- collect all samples ---
        for w in weathers:
            rgb_root = os.path.join(root_dir, "rgb_anon", w, split)
            gt_root = os.path.join(root_dir, "gt", w, split)
            if not os.path.exists(rgb_root):
                print(f"⚠️ Skip weather '{w}' (not found: {rgb_root})")
                continue

            for vid in os.listdir(rgb_root):
                img_dir = os.path.join(rgb_root, vid)
                mask_dir = os.path.join(gt_root, vid)
                # if not os.path.isdir(img_dir):
                #     continue

                for f in os.listdir(img_dir):
                    if f.endswith("_rgb_anon.png"):
                        base = f.replace("_rgb_anon.png", "")
                        mask_file = None
                        for suffix in ["_gt_labelTrainIds.png"]: #  "_gt_labelTrainIds.png", "_gt_labelIds.png"
                            candidate = os.path.join(mask_dir, base + suffix)
                            if os.path.exists(candidate):
                                mask_file = candidate
                                break
                        if mask_file:
                            self.samples.append((os.path.join(img_dir, f), mask_file))
                            # print(f"✅ Found mask for image: {mask_file}")
                        else:
                            print(f"⚠️ Missing mask for image: {os.path.join(img_dir, f)}")
                            

        print(f"✅ Loaded {len(self.samples)} ACDC {split} samples from weathers: {weathers}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]

        # --- Load image + mask ---
        image = Image.open(img_path).convert("RGB").resize(IMG_SIZE)
        mask = Image.open(mask_path).convert("L").resize(IMG_SIZE, 
                                                         resample=Image.NEAREST)

        image = np.array(image).transpose(2, 0, 1) / 255.0
        mask = np.array(mask, dtype=np.uint8)
        # # --- Optional transforms ---
        # if self.transform:
        #     image, mask = self.transform(image, mask)

        return torch.tensor(image, dtype=torch.float32), torch.tensor(mask, dtype=torch.long)


## Call Base Model

In [ ]:
def call_model(): 
    # ----- Reload base CLIP seg model to terminate Block-LoRA block -----
    clip_model = CLIPModel.from_pretrained(clip32_dir, use_safetensors=True)
    
    base_model = CLIPSegBaseline(
        clip_model,
        num_classes=NUM_CLASSES,
    ).to(DEVICE)
    
    if "pth" in MODEL_PATH:
        state_dict = torch.load(MODEL_PATH, 
                        map_location=DEVICE,
                        weights_only=False)["model_state_dict"] # True
    else:
        state_dict = torch.load(MODEL_PATH, 
                        map_location=DEVICE,
                        weights_only=False) # True
    
    decoder_state = {k.replace('decoder.', ''): v for k, v in state_dict.items() if 'decoder' in k}
    base_model.decoder.load_state_dict(decoder_state, strict=False)

    # # 🔧 เพิ่มความเร็วด้วย torch.compile() และ channels_last
    # base_model = torch.compile(base_model)
    # base_model.to(memory_format=torch.channels_last)
    # # base_model.to(memory_format=torch.contiguous_format)  

    print("Truly Reload Base Model")
    return base_model

In [ ]:
from torch.utils.data import DataLoader, ConcatDataset

BATCH_SIZE = 8
print("Full workesrs:", os.cpu_count())
mynum_workers = os.cpu_count() // 3
print(f"Using {mynum_workers} workers for DataLoader")

# ============================================================
# LOAD DATASETS ทั้งหมด (train / val / test)
# ============================================================
print("Training Dataset Loading ...")
muses_train = MusesDataset(muse_dir, split="train")
ade_train   = ADE20KDataset(ade20k_dir, split="train")
cs_train   = CityscapesDataset(cs_dir, split="train")
acdc_train  = ACDCDataset(acdc_dir, weather="all", split="train")

print('\nValidation Dataset Loading ...')
muses_val = MusesDataset(muse_dir, split="val")
ade_val   = ADE20KDataset(ade20k_dir, split="val")
cs_val   = CityscapesDataset(cs_dir, split="val")   
acdc_val  = ACDCDataset(acdc_dir, weather="all", split="val")
all_val = [muses_val, ade_val, cs_val, acdc_val]
# print("\nTest Dataset Loading ...")
# muses_test = MusesDataset(muse_dir, split="test")
# ade_test   = ADE20KDataset(ade20k_dir, split="test")  # ✅ ADE ไม่มี test จริง → fallback ไป val
# # cs_test   = CsDataset(cs_dir, split="test") 

# ============================================================
# รวม dataset เข้าด้วยกัน
# ============================================================
train_dataset = ConcatDataset([acdc_train, ade_train, cs_train, muses_train])
val_dataset   = ConcatDataset([acdc_val, ade_val, cs_val, muses_val])
# test_dataset  = ConcatDataset([muses_test, ade_test])
print('\nTotal Dataset Loaded')

def create_dataloader(trainset = None, validateaset = None):
    if trainset is None and validateaset is None:
        # ============================================================
        # สร้าง DataLoader
        # ============================================================
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, 
                                shuffle=True, num_workers=mynum_workers)
        val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, 
                                shuffle=False, num_workers=mynum_workers)
        return train_loader, val_loader

    if trainset is None:
        val_loader   = DataLoader(validateaset, batch_size=BATCH_SIZE, 
                                shuffle=False, num_workers=mynum_workers)
        return val_loader
    else:
        train_loader = DataLoader(trainset, batch_size=BATCH_SIZE, 
                                shuffle=True, num_workers=mynum_workers)
        val_loader   = DataLoader(validateaset, batch_size=BATCH_SIZE, 
                                shuffle=False, num_workers=mynum_workers)
        return train_loader, val_loader

train_loader = create_dataloader()
val_loader = create_dataloader()
# ============================================================
# แสดงสรุปผล
# ============================================================
print(f"✅ Train: {len(train_dataset)} samples")
print(f"✅ Val:   {len(val_dataset)} samples")



## Validate Model

In [ ]:

from torchmetrics.classification import MulticlassJaccardIndex
from tqdm.notebook import tqdm

@torch.no_grad()
def validate(model, val_loader, use_amp=True):
    model.eval()
    losses = AverageMeter()
    mious = AverageMeter()
    
    pbar = tqdm(val_loader, desc="Validation")
    
    for images, masks in pbar:
        images = images.to(DEVICE)
        masks = masks.to(DEVICE).long()
        
        if use_amp:
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, masks)
        else:
            outputs = model(images)
            loss = criterion(outputs, masks)
        
        pred = torch.argmax(outputs, dim=1)
        
        losses.update(loss.item(), images.size(0))
        miou = calculate_miou(pred, masks, NUM_CLASSES)
        mious.update(miou, images.size(0))
        
        pbar.set_postfix({'loss': f'{losses.avg:.4f}', 'mIoU': f'{mious.avg:.4f}'})
    
    return losses.avg, mious.avg

class AverageMeter:
    def __init__(self):
        self.reset()
    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

def calculate_miou(pred, target, num_classes=20, ignore_index=255):
    pred = pred.cpu().numpy()
    target = target.cpu().numpy()
    
    valid = (target != ignore_index)
    pred = pred[valid]
    target = target[valid]
    
    if len(pred) == 0:
        return 0.0
    
    ious = []
    present_classes = np.unique(target)
    
    for cls in present_classes:
        if cls == ignore_index:
            continue
        pred_cls = (pred == cls)
        target_cls = (target == cls)
        intersection = np.logical_and(pred_cls, target_cls).sum()
        union = np.logical_or(pred_cls, target_cls).sum()
        if union > 0:
            ious.append(intersection / union)
    
    return np.mean(ious) if ious else 0.0

def validate_model(model, val_loader):
    val_loss, val_miou = validate(model, val_loader)
    print(f"\n📊 Validation Summary:")
    print(f"  Val Loss: {val_loss:.4f}")
    print(f"  Val mIoU: {val_miou:.4f}")


## LoRA Validate

In [ ]:
lora_path = "/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/lora-db/ACDC-fog"
name = os.path.basename(os.path.abspath(os.path.join(lora_path, os.pardir)).lower())
print(f'Loading adapter from: {lora_path}\ndetected type: {name}')

MUSE+ACDC+ADE+CS
  Val Loss: 0.3957
  Val mIoU: 0.4221


['muses_val', 'i']
  Val Loss: 0.5155
  Val mIoU: 0.3569


['ade_val', 'i']
  Val Loss: 0.3648
  Val mIoU: 0.4662


['cs_val', 'i']
  Val Loss: 0.3792
  Val mIoU: 0.3997


['acdc_val', 'i']
  Val Loss: 0.4065
  Val mIoU: 0.4046

In [ ]:
def namestr(obj, namespace):
    return [name for name in namespace if namespace[name] is obj]

In [ ]:
model = call_model()
model.eval()
print("\nValidating original model:")
_, dset = create_dataloader()
validate_model(model, dset)
for i in all_val:
    print("\n")
    print(namestr(i, globals()))
    dset = create_dataloader(validateaset = i)
    validate_model(model, dset)
    # print(type(model.clip))

In [ ]:
model2 = call_model()

lora_path = "/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/6base16bz/blora-db/ADE20K"
# config = BlockLoraConfig(**cfg_dict)
model2.clip = BlockLoraModel.from_pretrained(model2.clip,  lora_path).model
model2.clip.to(DEVICE)
# model2.clip.merge_and_unload()
for i in all_val:
    print("\n\n")
    print(namestr(i, globals()))
    dset = create_dataloader(validateaset = i)
    validate_model(model2,dset)

In [ ]:
model2 = call_model()

lora_path = "/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/6base16bz/blora-db/CS"
# config = BlockLoraConfig(**cfg_dict)
model2.clip = BlockLoraModel.from_pretrained(model2.clip,  lora_path).model
model2.clip.to(DEVICE)
# model2.clip.merge_and_unload()
for i in all_val:
    print("\n\n")
    print(namestr(i, globals()))
    dset = create_dataloader(validateaset = i)
    validate_model(model2,dset)

In [ ]:
a=BlockLoraModel.from_pretrained(model.clip,  lora_path).model

In [ ]:
a

In [ ]:
model3 = call_model()
# q_weight_before = model3.clip.transformer.resblocks[0].attn.out_proj.weight
lora_path = "/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/lora-db/MUSES_clear"
model3.clip = PeftModel.from_pretrained(model3.clip, lora_path)

print("Validating LoRA")
for i in all_val:
    print("\n\n")
    print(namestr(i, globals()))
    dset = create_dataloader(validateaset = i)
    validate_model(model3,dset)
# q_weight_after = model3.clip.transformer.resblocks[0].attn.out_proj.weight
# print("Are out_proj weights changed?", not torch.allclose(q_weight_before, q_weight_after))


In [ ]:
type(model3)

In [ ]:
import torch

def extract_lora_weights(base_weight: torch.Tensor, merged_weight: torch.Tensor, rank: int):
    """
    Extract LoRA weights (A, B) from merged weight and base weight.
    Args:
        base_weight: torch.Tensor of shape [out_dim, in_dim]
        merged_weight: torch.Tensor of shape [out_dim, in_dim] (W + LoRA)
        rank: low-rank r of LoRA

    Returns:
        A: torch.Tensor of shape [out_dim, rank]
        B: torch.Tensor of shape [rank, in_dim]
        delta_W: full LoRA weight (W' - W)
    """
    # 1️⃣ Compute delta_W
    delta_W = merged_weight - base_weight

    # 2️⃣ Low-rank SVD approximation
    U, S, Vh = torch.linalg.svd(delta_W, full_matrices=False)

    # 3️⃣ Keep top-r components
    U_r = U[:, :rank]        # [out_dim, r]
    S_r = S[:rank]           # [r]
    Vh_r = Vh[:rank, :]      # [r, in_dim]

    # 4️⃣ Form LoRA factors
    A = U_r * torch.sqrt(S_r)[None, :]      # broadcasting sqrt(S_r)
    B = torch.sqrt(S_r)[:, None] * Vh_r

    return A, B, delta_W

# =============================
# Example usage
# =============================
# Suppose q_proj.weight of base and merged model
# base_weight = base_model.clip.transformer.resblocks[0].attn.q_proj.weight
# merged_weight = merged_model.clip.transformer.resblocks[0].attn.q_proj.weight

# # Choose LoRA rank (r)
# rank = 4

# A, B, delta_W = extract_lora_weights(base_weight, merged_weight, rank)

# print("Delta W shape:", delta_W.shape)
# print("A shape:", A.shape)
# print("B shape:", B.shape)
